# Appendix Figure Generation

This notebook generates comprehensive supplementary figures for the PEARL TB screening manuscript appendix.

**Content:**
- Extended calibration figures for multiple task configurations
- Posterior vs prior distributions across parameter sensitivity analyses
- Diff outputs for all screening scenarios
- Trajectory comparisons for all scenario pairs
- Convergence diagnostics (R-hat) by configuration

All figures are organized by configuration parameter (relative susceptibility and clinical regression rate).

In [ ]:
# Import standard libraries
from pathlib import Path
from math import ceil
import numpy as np
import pandas as pd
import yaml

# Import scientific/plotting
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Import project modules
from tbh.paths import REPO_ROOT_PATH
from tbh.model import get_tb_model
from tbh.plotting import plot_model_fit_with_uncertainty, plot_two_scenarios, plot_diff_outputs, title_lookup
import tbh.plotting as pl
import tbh.runner_tools as rt
from estival.model import BayesianCompartmentalModel

# Configure matplotlib
plt.style.use("seaborn-v0_8-white")
text_color = "#1f1f1f"
plt.rcParams.update({
    "font.family": "Helvetica",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "legend.frameon": False,
    "legend.fontsize": 7,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "text.color": text_color,
    "axes.labelcolor": text_color,
    "axes.titlecolor": text_color,
    "xtick.color": text_color,
    "ytick.color": text_color,
    "axes.edgecolor": text_color,
})

print("Imports successful")

## Configuration and Setup

In [ ]:
# Set paths
BASE_DIR = REPO_ROOT_PATH / "remote_cluster" / "outputs" / "59094989_new_priors"
OUTPUT_DIR = REPO_ROOT_PATH / "notebooks" / "manuscript_figs" / "appendix"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Intervention scenarios
INTERVENTION_SCENARIOS = [sc.sc_id for sc in rt.SCENARIOS]

# Trajectory outputs for appendix
TRAJECTORY_OUTPUTS = [
    "tb_incidence_per100k",
    "viable_tbi_prevalence_perc",
    "tb_prevalence_per100k",
    "tb_mortality_per100k",
]

SC_NAMES_CUSTOM = {
    'baseline': 'No screening',
    'scenario_1': 'PEARL / 65%',
    'scenario_2': 'PEARL / 75%',
    'scenario_3': 'PEARL / 85%',
    'scenario_6': 'CXR-TST / 65%',
    'scenario_7': 'CXR-TST / 75%',
    'scenario_8': 'CXR-TST / 85%',
    'scenario_16': 'CXR / 65%',
    'scenario_17': 'CXR / 75%',
    'scenario_18': 'CXR / 85%'
}

print(f"Output directory: {OUTPUT_DIR}")
print(f"Number of intervention scenarios: {len(INTERVENTION_SCENARIOS)}")

## Load all task data

In [ ]:
# Scan directory for completed tasks
task_dirs = sorted([p for p in BASE_DIR.iterdir() if p.is_dir()])

records = []
for task_dir in task_dirs:
    files = [x for x in task_dir.iterdir() if x.is_file()]
    names = {f.name for f in files}
    
    has_idata = "idata.nc" in names
    has_details = "details.yaml" in names
    
    if len(files) == 0:
        status = "empty"
    elif has_details:
        status = "completed"
    elif has_idata:
        status = "partial"
    else:
        status = "failed"
    
    records.append({
        "task": task_dir.name,
        "task_path": task_dir,
        "n_files": len(files),
        "status": status,
    })

status_df = pd.DataFrame(records)
completed_df = status_df.loc[status_df["status"] == "completed"].copy()
print(f"Found {len(completed_df)} completed tasks")

In [ ]:
# Load configuration map
config_map_path = BASE_DIR / "task_config_map.yaml"
with open(config_map_path, "r") as f:
    raw_task_config_map = yaml.safe_load(f)

config_source = raw_task_config_map.get("tasks", raw_task_config_map) if isinstance(raw_task_config_map, dict) else raw_task_config_map
task_to_config = config_source.get("task_to_config", {}) if isinstance(config_source, dict) else {}

def get_task_config(task_name):
    task_num = int(task_name.split("_")[1])
    return task_to_config.get(task_num, task_to_config.get(str(task_num)))

def get_config_label(task_cfg):
    if not isinstance(task_cfg, dict):
        return "unknown"
    rel_sus = task_cfg.get("rel_sus_unreachable", "na")
    reg = task_cfg.get("clinical_regression_rate", "na")
    return f"rel_sus={rel_sus}, reg={reg}"

completed_df["task_config"] = completed_df["task"].apply(get_task_config)
completed_df["config_label"] = completed_df["task_config"].apply(get_config_label)

# Group by configuration
config_groups = completed_df.groupby("config_label")
print(f"Tasks organized into {len(config_groups)} configuration groups")
print("\nConfiguration groups:")
for label, group in config_groups:
    print(f"  {label}: {len(group)} task(s)")

## Generate appendix figures by configuration

In [ ]:
# Helper function to load task bundle
def load_task_bundle(task_row):
    task_path = Path(task_row["task_path"])
    task_name = task_row["task"]
    cfg_label = task_row["config_label"]
    
    required_files = [
        task_path / "idata.nc",
        task_path / "details.yaml",
        task_path / "uncertainty_df_baseline.parquet",
    ]
    
    missing = [p.name for p in required_files if not p.exists()]
    if missing:
        return None
    
    try:
        idata = az.from_netcdf(task_path / "idata.nc")
        
        with open(task_path / "details.yaml", "r") as f:
            docs = list(yaml.safe_load_all(f))
        
        model_config = docs[1] if len(docs) > 1 and isinstance(docs[1], dict) else {}
        analysis_config = docs[2] if len(docs) > 2 and isinstance(docs[2], dict) else {}
        
        params, priors, tv_params = rt.get_parameters_and_priors()
        model = get_tb_model(model_config, tv_params)
        bcm = BayesianCompartmentalModel(model, params, priors, rt.targets)
        
        # Load all available uncertainty dataframes
        unc_dfs = {}
        for unc_file in task_path.glob("uncertainty_df_*.parquet"):
            scenario = unc_file.name.replace("uncertainty_df_", "").replace(".parquet", "")
            unc_dfs[scenario] = pd.read_parquet(unc_file)
        
        # Load all available diff dataframes
        diff_dfs = {}
        for diff_file in task_path.glob("diff_quantiles_df_ref_baseline_*.parquet"):
            scenario = diff_file.name.replace("diff_quantiles_df_ref_baseline_", "").replace(".parquet", "")
            diff_dfs[scenario] = pd.read_parquet(diff_file)
        
        return {
            "task_name": task_name,
            "task_path": task_path,
            "cfg_label": cfg_label,
            "idata": idata,
            "bcm": bcm,
            "unc_dfs": unc_dfs,
            "diff_dfs": diff_dfs,
            "analysis_config": analysis_config,
        }
    except Exception as e:
        print(f"Error loading {task_name}: {e}")
        return None

print("Task loading function defined")

In [ ]:
# Generate calibration figures for each configuration (select first task per config)
from IPython.display import display, Markdown

for config_label, config_group in config_groups:
    # Use first task in configuration group
    task_row = config_group.iloc[0]
    bundle = load_task_bundle(task_row)
    
    if bundle is None:
        print(f"Skipping {config_label}: unable to load task")
        continue
    
    display(Markdown(f"## {config_label}"))
    display(Markdown(f"Task: {bundle['task_name']}"))
    
    # Generate calibration figure
    if "baseline" not in bundle["unc_dfs"]:
        display(Markdown("*Calibration figure skipped: baseline uncertainty file missing*"))
        continue
    
    # Create calibration plot (simplified version for appendix)
    selected_outputs = [
        "pearl_posXreach_reachable_per100k",
        "cxr_posXreach_reachable_per100k",
        "tb_prevalence_per100k",
        "tb_incidence_per100k",
        "notifications",
    ]
    
    n_col = 3
    n_panels = len(selected_outputs)
    n_row = ceil(n_panels / n_col)
    
    fig, axes = plt.subplots(n_row, n_col, figsize=(10, 3.2 * n_row))
    axes = axes.flatten()
    
    unc_df = bundle["unc_dfs"]["baseline"]
    bcm = bundle["bcm"]
    
    for i, output in enumerate(selected_outputs):
        ax = axes[i]
        x_min = 1990 if output == "notifications" else 2010
        plot_model_fit_with_uncertainty(ax, unc_df, output, bcm, x_lim=(x_min, 2025), colour="#B22222")
    
    for j in range(len(selected_outputs), len(axes)):
        fig.delaxes(axes[j])
    
    fig.suptitle(f"Calibration: {config_label}", y=1.00, fontsize=11)
    fig.tight_layout()
    
    # Save figure
    safe_label = config_label.replace(", ", "_").replace("=", "-")
    fig_path = OUTPUT_DIR / f"appendix_calibration_{safe_label}"
    plt.savefig(f"{fig_path}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{fig_path}.pdf", bbox_inches='tight')
    
    display(fig)
    plt.close(fig)
    
    display(Markdown("---"))

In [ ]:
# Generate diff outputs for all scenarios (select base-case task)
base_task_row = completed_df.iloc[0]  # First completed task
base_bundle = load_task_bundle(base_task_row)

if base_bundle and base_bundle["diff_dfs"]:
    display(Markdown("## Diff outputs - all scenarios"))
    display(Markdown(f"Task: {base_bundle['task_name']} ({base_bundle['cfg_label']})"))
    
    available_scenarios = list(base_bundle["diff_dfs"].keys())
    scenarios_in_order = [s for s in ["scenario_1", "scenario_2", "scenario_3", "scenario_6", 
                                       "scenario_7", "scenario_8", "scenario_16", "scenario_17", 
                                       "scenario_18"] if s in available_scenarios]
    
    diff_outputs = ["TB_averted_relative", "deaths_averted_relative"]
    
    for output in diff_outputs:
        fig, ax = plt.subplots(figsize=(8, 3))
        
        try:
            plot_diff_outputs(ax, base_bundle["diff_dfs"], output, scenarios_in_order, sc_names=SC_NAMES_CUSTOM)
            plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
            ax.grid(axis="y", linestyle="-", linewidth=0.7, alpha=0.4)
            
            fig.suptitle(f"Appendix: {title_lookup.get(output, output)}", y=1.00, fontsize=11)
            fig.tight_layout()
            
            safe_output = output.replace("_", "-")
            fig_path = OUTPUT_DIR / f"appendix_diff_{safe_output}"
            plt.savefig(f"{fig_path}.png", dpi=300, bbox_inches='tight')
            plt.savefig(f"{fig_path}.pdf", bbox_inches='tight')
            
            display(fig)
        except Exception as e:
            print(f"Error plotting {output}: {e}")
        finally:
            plt.close(fig)
    
    display(Markdown("---"))
else:
    print("Could not load base task for diff outputs")

In [ ]:
# Generate trajectory comparisons for selected scenario pairs
display(Markdown("## Trajectory comparisons - all scenarios vs baseline"))

if base_bundle and base_bundle["unc_dfs"]:
    scenario_pairs = [
        ("baseline", "scenario_3"),
        ("baseline", "scenario_8"),
        ("baseline", "scenario_18"),
    ]
    
    unc_dfs = base_bundle["unc_dfs"]
    available_scenarios = set(unc_dfs.keys())
    
    unc_sc_colours = ["#200895", "#9a0e52"]
    
    for baseline_sc, compare_sc in scenario_pairs:
        if baseline_sc not in available_scenarios or compare_sc not in available_scenarios:
            continue
        
        fig, axes = plt.subplots(2, 2, figsize=(8, 5.5))
        axes = axes.flatten()
        
        for ax, output in zip(axes, TRAJECTORY_OUTPUTS):
            plot_two_scenarios(
                ax,
                unc_dfs,
                output,
                scenarios=[baseline_sc, compare_sc],
                xlim=(2020, 2035),
                include_unc=True,
                ylab_fontsize=9,
                unc_sc_colours=unc_sc_colours,
                include_legend=ax==axes[0],
                sc_names=SC_NAMES_CUSTOM
            )
            ax.set_title(title_lookup.get(output, output), fontsize=10)
        
        fig.suptitle(f"Appendix: {SC_NAMES_CUSTOM[compare_sc]} vs {SC_NAMES_CUSTOM[baseline_sc]}", 
                     y=1.00, fontsize=11)
        fig.tight_layout()
        
        safe_label = f"{compare_sc}_vs_{baseline_sc}"
        fig_path = OUTPUT_DIR / f"appendix_trajectories_{safe_label}"
        plt.savefig(f"{fig_path}.png", dpi=300, bbox_inches='tight')
        plt.savefig(f"{fig_path}.pdf", bbox_inches='tight')
        
        display(fig)
        plt.close(fig)

## Summary

In [ ]:
# Generate index of all figures created
from pathlib import Path

png_files = sorted(OUTPUT_DIR.glob("*.png"))
pdf_files = sorted(OUTPUT_DIR.glob("*.pdf"))

print("\n" + "="*70)
print("APPENDIX FIGURE GENERATION COMPLETE")
print("="*70)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nTotal figures generated: {len(png_files)}")
print("\nFigure list:")

for png_file in png_files:
    print(f"  • {png_file.stem}")

print(f"\nAll figures saved in PNG (300 dpi) and PDF formats.")
print(f"\nNext step: Create quarto document to compile these figures into appendix.")